In [0]:
catalog, schema = "workspace", "dm2_project"

# Objective 1: revenue + orders by region over time
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.gold_revenue_by_region AS
SELECT region, invoice_month,
       ROUND(SUM(line_revenue), 2) AS revenue,
       COUNT(DISTINCT invoice_no)  AS orders
FROM {catalog}.{schema}.silver_sales_enriched
GROUP BY region, invoice_month
""")

# Objective 2: top products by revenue
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.gold_top_products AS
SELECT stock_code,
       MAX(description)            AS description,
       ROUND(SUM(line_revenue), 2) AS revenue,
       SUM(quantity)               AS units
FROM {catalog}.{schema}.silver_sales_enriched
GROUP BY stock_code
ORDER BY revenue DESC
LIMIT 20
""")

print("gold tables built")
print("--- Objective 1: revenue by region ---")
display(spark.table(f"{catalog}.{schema}.gold_revenue_by_region"))
print("--- Objective 2: top products ---")
display(spark.table(f"{catalog}.{schema}.gold_top_products"))